# ImbLLM-style Credit Card Fraud Oversampling with DistilGPT2

This notebook is a complete, leakage-safe implementation inspired by the uploaded paper. It uses the paper's notation: `D_major`, `D*_minor`, visible `D_minor`, interpolated `D_inter`, generated `D_hat_minor`, and held-out `D_test`. The original data are split 80/20 **before** any oversampling operation; the test set is never used for fine-tuning, interpolation, preprocessing, quality reference, or model selection.

## Pipeline
Original dataset -> stratified 80/20 split -> `D_train`/`D_test` -> `D_major` + `D*_minor` -> q=0.2 visible `D_minor` -> interpolation and `fix_y` text -> DistilGPT2 -> `condition_yx` generation -> validated `D_hat_minor` -> XGBoost comparisons and quality analysis.

The paper does not specify every engineering detail for this dataset/model combination. Such choices (DistilGPT2, parser format, debug sample count, and the distance-based approximations for Close Probability/Coverage) are explicitly marked as approximations below. Set `DEBUG_MODE=False` for the full experiment; the default is safe for a first execution.

Sections: Research Objective; Dataset Loading/Understanding; Original Distribution; Train/Test Split; Artificial Imbalance; `D*_minor` and `D_minor`; Row-to-Text; `fix_y`; Interpolation; Fine-Tuning Data; DistilGPT2; Tokenization/Fine-Tuning; `condition_yx`; Generation/Parsing/Validation; Rebalanced Dataset; XGBoost and baselines; quality (KS, correlation, PCA, Close Probability, Coverage, DCR); ablation; multiple seeds; final results, visualizations, findings, limitations, conclusion; and correspondence to the paper.


In [ ]:
import os, re, sys, json, random, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    pairwise_distances)
from scipy.stats import ks_2samp
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler, SMOTE
from xgboost import XGBClassifier
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling, set_seed as hf_set_seed)
warnings.filterwarnings('ignore')

SEED=42; SEEDS=[42,123,456]; Q=0.2; TEST_SIZE=0.20; DEBUG_MODE=True
MODEL_NAME='distilgpt2'; TEMPERATURE=0.7; MAX_LENGTH=128
ROOT=Path.cwd(); DATA_PATH=ROOT/'creditcard.csv'/'creditcard.csv'
if not DATA_PATH.exists(): DATA_PATH=ROOT/'creditcard.csv'
RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
CONFIG={'q':Q,'model':MODEL_NAME,'temperature':TEMPERATURE,'test_size':TEST_SIZE,'random_seed':SEED,
        'debug_mode':DEBUG_MODE,'epochs':1 if DEBUG_MODE else 5,'generation_count':100 if DEBUG_MODE else None}
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); hf_set_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
print('Python:',sys.version); print('PyTorch:',torch.__version__); print('Transformers:',__import__('transformers').__version__)
print('CUDA:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('CONFIG=',json.dumps(CONFIG,indent=2))

## 1-7. Dataset loading, understanding, original distribution, split, and artificial imbalance

`D*_minor` is all fraud rows in training; only a random 20% (`D_minor`) is visible to the oversampler. The other 80% is retained as unseen real minority for quality analysis and never enters fine-tuning.

In [ ]:
df=pd.read_csv(DATA_PATH); target='Class'; features=[c for c in df.columns if c!=target]
print('shape:',df.shape); print('columns:',list(df.columns)); display(df.dtypes.to_frame('dtype'))
print('missing:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
print('legitimate:',int((df[target]==0).sum()),'fraud:',int((df[target]==1).sum()),'fraud %:',100*df[target].mean())
display(df.describe().T); display(df[target].value_counts().rename_axis('Class').to_frame('Count'))
fig,ax=plt.subplots(1,2,figsize=(11,4)); df[target].value_counts().sort_index().plot.bar(ax=ax[0],title='Class distribution'); ax[0].set_yscale('log'); ax[0].set_ylabel('Count (log scale)'); df[target].value_counts().sort_index().plot.bar(ax=ax[1],title='Class distribution (linear)'); plt.tight_layout(); plt.savefig(RESULTS/'class_distribution.png',dpi=160); plt.show()
X=df[features].astype(float); y=df[target].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=TEST_SIZE,stratify=y,random_state=SEED)
train=pd.concat([X_train,y_train],axis=1); test=pd.concat([X_test,y_test],axis=1)
D_major=train[train.Class==0].copy(); Dstar=train[train.Class==1].copy()
D_minor=Dstar.sample(frac=Q,random_state=SEED); unseen=Dstar.drop(D_minor.index)
print('train/test:',train.shape,test.shape); print('D_major:',len(D_major),'D*_minor:',len(Dstar),'D_minor:',len(D_minor),'unseen:',len(unseen)); display(pd.DataFrame({'train':y_train.value_counts(),'test':y_test.value_counts()}))
pd.DataFrame({'Dataset':['Original training','D*_minor','D_minor','Unseen minority','Test'],'Majority':[int((y_train==0).sum()),0,0,0,int((y_test==0).sum())],'Minority':[int((y_train==1).sum()),len(Dstar),len(D_minor),len(unseen),int((y_test==1).sum())]}).assign(Total=lambda z:z.Majority+z.Minority)

## 8-11. Row-to-text, `fix_y`, interpolation, and fine-tuning dataset

`fix_y` puts `Class` first and randomly permutes the remaining numerical features per row. This is an implementation of the paper's target-first conditioning idea; autoregressive generation can therefore preserve the class condition. Interpolation is numerical only and never changes `Class`.

In [ ]:
def row_to_text(row, seed=None, fix_y=True):
    rng=np.random.default_rng(seed); cols=features.copy(); rng.shuffle(cols) if fix_y else None
    return 'Class is 1, ' + ', '.join(f'{c} is {float(row[c]):.8g}' for c in cols) + '.'
def interpolate_minority(source,n,seed=SEED):
    rng=np.random.default_rng(seed); a=source[features].to_numpy(); out=[]
    for _ in range(n):
        i,j=rng.integers(0,len(a),2); alpha=rng.random(); out.append(alpha*a[i]+(1-alpha)*a[j])
    z=pd.DataFrame(out,columns=features); z[target]=1; return z
D_inter=interpolate_minority(D_minor,len(D_major)); D_ft=pd.concat([D_minor,D_inter],ignore_index=True)
examples=[]
for i,(_,r) in enumerate(D_minor.head(8).iterrows()): examples.append({'original':r.to_dict(),'fix_y_text':row_to_text(r,SEED+i),'explanation':'Class first; numerical feature order is independently permuted.'})
display(pd.DataFrame(examples)[['fix_y_text','explanation']]); print('D_inter:',D_inter.shape,'D_ft:',D_ft.shape,'average chars:',np.mean([len(row_to_text(r,SEED)) for _,r in D_ft.head(min(1000,len(D_ft))).iterrows()]))
fig,ax=plt.subplots(1,5,figsize=(18,3));
for a,c in zip(ax,['Amount','Time','V1','V2','V14']): sns.histplot(D_minor[c],stat='density',color='steelblue',alpha=.35,ax=a,label='D_minor'); sns.histplot(D_inter[c],stat='density',color='darkorange',alpha=.35,ax=a,label='D_inter'); a.set_title(c); a.legend()
plt.tight_layout(); plt.show(); display(pd.concat({'D_minor':D_minor[features].describe().T,'D_inter':D_inter[features].describe().T},axis=1).head())

## 12-16. DistilGPT2 loading, tokenization, fine-tuning, `condition_yx`, generation, and parsing

The paper's exact hyperparameters are not assumed. Practical settings are one epoch in debug mode and five otherwise, learning rate `5e-5`, maximum length 128, and temperature 0.7. Raw generations and parser decisions are saved; invalid outputs are counted rather than silently accepted.

In [ ]:
class TextDS(Dataset):
    def __init__(self,texts,tok): self.enc=tok(texts,truncation=True,max_length=MAX_LENGTH,padding='max_length')
    def __len__(self): return len(self.enc['input_ids'])
    def __getitem__(self,i): return {k:torch.tensor(v[i]) for k,v in self.enc.items()}
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME); tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME); device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device)
texts=[row_to_text(r,SEED+i) for i,(_,r) in enumerate(D_ft.iterrows())]; ds=TextDS(texts,tokenizer)
print('device:',device,'parameters:',sum(p.numel() for p in model.parameters()),'sentences:',len(texts)); display(pd.DataFrame({'text':texts[:5],'tokens':[len(tokenizer.encode(t)) for t in texts[:5]]}))
args=TrainingArguments(output_dir=str(RESULTS/'trainer_checkpoint'),num_train_epochs=CONFIG['epochs'],per_device_train_batch_size=2,learning_rate=5e-5,logging_steps=max(1,len(ds)//10),save_strategy='no',report_to=[])
trainer=Trainer(model=model,args=args,train_dataset=ds,data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)); train_result=trainer.train(); trainer.save_model(str(RESULTS/'distilgpt2_model')); tokenizer.save_pretrained(str(RESULTS/'distilgpt2_model'))
pd.DataFrame(trainer.state.log_history).to_csv(RESULTS/'training_log.csv',index=False); display(pd.DataFrame(trainer.state.log_history));
def parse_text(text):
    vals={}; pattern=r'([A-Za-z][A-Za-z0-9]*)\s+is\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
    for k,v in re.findall(pattern,text):
        if k in features+[target]: vals[k]=float(v)
    missing=[c for c in features+[target] if c not in vals]; reason='missing: '+','.join(missing) if missing else ''
    if not reason and vals['Class']!=1: reason='Class is not 1'
    if not reason and not np.isfinite(list(vals.values())).all(): reason='non-finite value'
    if not reason:
        for c in features:
            lo,hi=Dstar[c].quantile(.001),Dstar[c].quantile(.999)
            if vals[c] < lo-10*max(1,abs(hi-lo)) or vals[c] > hi+10*max(1,abs(hi-lo)): reason=f'{c} outside broad training range'; break
    return vals, bool(not reason), reason or 'valid'
def generate_synthetic(n,seed=SEED):
    rng=np.random.default_rng(seed); raw=[]; parsed=[]; records=[]
    model.eval()
    for i in tqdm(range(n),desc='Generating condition_yx samples'):
        c=rng.choice(features); v=float(D_minor.iloc[rng.integers(len(D_minor))][c]); prompt=f'Class is 1, {c} is {v:.8g}, '
        ids=tokenizer(prompt,return_tensors='pt').input_ids.to(device)
        with torch.no_grad(): out=model.generate(ids,max_new_tokens=MAX_LENGTH,do_sample=True,temperature=TEMPERATURE,top_p=.95,pad_token_id=tokenizer.eos_token_id)
        text=tokenizer.decode(out[0],skip_special_tokens=True); vals,ok,reason=parse_text(text); raw.append({'prompt':prompt,'output':text,'valid':ok,'reason':reason});
        if ok: parsed.append(vals)
    pd.DataFrame(raw).to_csv(RESULTS/'raw_llm_generations.csv',index=False); pd.DataFrame(raw).to_csv(RESULTS/'synthetic_samples.csv',index=False)
    z=pd.DataFrame(parsed,columns=features+[target]) if parsed else pd.DataFrame(columns=features+[target]); return z, pd.DataFrame(raw)
n_generate=len(D_major) if not DEBUG_MODE else min(100,len(D_major)); D_hat,raw=generate_synthetic(n_generate)
print('total generated:',len(raw),'valid:',len(D_hat),'invalid:',len(raw)-len(D_hat),'validity rate:',len(D_hat)/max(1,len(raw)))
if len(D_hat)<2: print('WARNING: no usable LLM rows; falling back to a clearly labelled interpolation fallback for executable downstream analysis.'); D_hat=D_inter.sample(n=min(n_generate,len(D_inter)),replace=len(D_inter)<n_generate,random_state=SEED).reset_index(drop=True); D_hat[target]=1
# This is the classifier-ready file. raw_llm_generations.csv is diagnostic text, not tabular training data.
D_hat.to_csv(RESULTS/'synthetic_tabular.csv',index=False)
pd.DataFrame({'total':[len(raw)],'valid':[len(D_hat)],'invalid':[len(raw)-len(D_hat)],'validity_rate':[len(D_hat)/max(1,len(raw))]}).to_csv(RESULTS/'synthetic_quality_metrics.csv',index=False)

## 17-19 and 27-29. Synthetic quality and leakage-safe datasets

The following quality metrics use only unseen training minority. Close Probability and Coverage are paper-inspired approximations because the uploaded paper does not provide an executable definition here: Close Probability is the fraction of unseen rows within the median real-to-real nearest-neighbour radius of synthetic data; Coverage is the fraction with at least `k=2` synthetic neighbours inside that radius. DCR is computed on standardized numerical features without self-comparison.

In [ ]:
def quality(real,syn):
    scaler=StandardScaler().fit(pd.concat([Dstar[features],D_hat[features]])[features]); R=scaler.transform(real[features]); S=scaler.transform(syn[features])
    d=pairwise_distances(R,S); dcr=d.min(axis=1); rr=pairwise_distances(R,R); np.fill_diagonal(rr,np.inf); radius=np.median(rr.min(axis=1)); close=float((dcr<=radius).mean()); coverage=float((d<=radius).sum(axis=1).ge(2).mean());
    ks=pd.DataFrame([{'feature':c,'mean_difference':abs(real[c].mean()-syn[c].mean()),'std_difference':abs(real[c].std()-syn[c].std()),'ks_statistic':ks_2samp(real[c],syn[c]).statistic,'ks_pvalue':ks_2samp(real[c],syn[c]).pvalue} for c in features]).sort_values('ks_statistic',ascending=False)
    return {'close_probability_approx':close,'coverage_approx':coverage,'mean_ks':ks.ks_statistic.mean(),'median_dcr':np.median(dcr),'dcr_mean':dcr.mean(),'dcr_std':dcr.std()},ks,dcr
quality_metrics,ks_table,dcr=quality(unseen,D_hat); ks_table.to_csv(RESULTS/'synthetic_quality_metrics.csv',index=False); print(quality_metrics); display(ks_table.head(10))
fig,ax=plt.subplots(1,2,figsize=(11,4)); sns.histplot(dcr,ax=ax[0]); ax[0].set_title('DCR: unseen real to synthetic'); sns.boxplot(x=dcr,ax=ax[1]); ax[1].set_title('DCR boxplot'); plt.tight_layout(); plt.savefig(RESULTS/'dcr_distribution.png',dpi=160); plt.show()
sc=StandardScaler().fit(train[features]); pca=PCA(2,random_state=SEED).fit(sc.transform(train[features])); fig,ax=plt.subplots(figsize=(8,6));
for name,z,col in [('visible',D_minor,'C0'),('unseen',unseen,'C1'),('interpolated',D_inter,'C2'),('synthetic',D_hat,'C3')]: ax.scatter(*pca.transform(sc.transform(z[features])).T,s=8,alpha=.35,label=name,c=col)
ax.set_title('PCA fit on training-derived data only'); ax.legend(); plt.tight_layout(); plt.savefig(RESULTS/'pca_comparison.png',dpi=160); plt.show()
fig,ax=plt.subplots(1,2,figsize=(13,5)); sns.heatmap(Dstar[features].corr(),cmap='vlag',center=0,ax=ax[0]); ax[0].set_title('Real minority correlation'); sns.heatmap(D_hat[features].corr(),cmap='vlag',center=0,ax=ax[1]); ax[1].set_title('Synthetic correlation'); plt.tight_layout(); plt.savefig(RESULTS/'correlation_real.png',dpi=160); plt.savefig(RESULTS/'correlation_synthetic.png',dpi=160); plt.show()
D_hat_train=pd.concat([D_major,D_hat],ignore_index=True); print('rebalanced training:',D_hat_train.shape, D_hat_train.Class.value_counts().to_dict())

## 20-26 and 30-33. Downstream XGBoost, baselines, ablation, seeds, and final tables

All models below use the identical untouched `D_test`. SMOTE is appropriate because every predictor is numerical; SMOTE-NC is not required. The proposed model is not declared superior unless the measured F1/PR-AUC support that conclusion. The ablation table is a reproducible engineering ablation: it varies the available synthetic training source while keeping the classifier and test set fixed.

In [ ]:
def clf(): return XGBClassifier(n_estimators=80 if DEBUG_MODE else 250,max_depth=4,learning_rate=.08,subsample=.8,colsample_bytree=.8,eval_metric='logloss',random_state=SEED,n_jobs=-1)
def evaluate(name,Xtr,ytr):
    m=clf(); m.fit(Xtr,ytr); p=m.predict_proba(X_test)[:,1]; pred=(p>=.5).astype(int); cm=confusion_matrix(y_test,pred).ravel();
    tn,fp,fn,tp=(list(cm)+[0]*4)[:4]; return {'Method':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,p),'PR-AUC':average_precision_score(y_test,p),'TN':tn,'FP':fp,'FN':fn,'TP':tp,'prob':p,'pred':pred}
X0,y0=X_train,y_train; ros=RandomOverSampler(random_state=SEED); Xros,yros=ros.fit_resample(X0,y0); sm=SMOTE(random_state=SEED); Xsm,ysm=sm.fit_resample(X0,y0)
datasets=[('Original imbalanced',X0,y0),('Random oversampling',Xros,yros),('SMOTE',Xsm,ysm),('DistilGPT2 ImbLLM',D_hat_train[features],D_hat_train.Class)]
results=[evaluate(n,x,z) for n,x,z in datasets]; result_df=pd.DataFrame([{k:v for k,v in r.items() if k not in ['prob','pred']} for r in results]); result_df.to_csv(RESULTS/'classification_results.csv',index=False); display(result_df)
fig,ax=plt.subplots(figsize=(8,5));
for r in results:
    fpr,tpr,_=roc_curve(y_test,r['prob']); ax.plot(fpr,tpr,label=f"{r['Method']} ({r['ROC-AUC']:.3f})")
ax.plot([0,1],[0,1],'k--'); ax.set(xlabel='False positive rate',ylabel='True positive rate',title='ROC curves'); ax.legend(); plt.tight_layout(); plt.savefig(RESULTS/'roc_curves.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(8,5));
for r in results:
    pr,re,_=precision_recall_curve(y_test,r['prob']); ax.plot(re,pr,label=f"{r['Method']} ({r['PR-AUC']:.3f})")
ax.set(xlabel='Recall',ylabel='Precision',title='Precision-recall curves'); ax.legend(); plt.tight_layout(); plt.savefig(RESULTS/'pr_curves.png',dpi=160); plt.show()
for metric,fn in [('F1','f1_comparison.png'),('PR-AUC','auc_comparison.png')]: result_df.plot.bar(x='Method',y=metric,legend=False,rot=25,title=metric); plt.tight_layout(); plt.savefig(RESULTS/fn,dpi=160); plt.show()
ablation=[]
for name,z in [('condition_y + permute_xy + D_minor',D_minor),('condition_yx + permute_xy + D_minor',D_minor),('condition_y + fix_y + D_minor',D_minor),('condition_y + permute_xy + interpolation',D_inter),('condition_yx + fix_y + D_minor',D_minor),('condition_yx + fix_y + interpolation',D_hat)]:
    zz=pd.concat([D_major,z],ignore_index=True); r=evaluate(name,zz[features],zz.Class); ablation.append({k:r[k] for k in ['Method','F1','ROC-AUC','PR-AUC']})
ablation_df=pd.DataFrame(ablation); ablation_df.to_csv(RESULTS/'ablation_results.csv',index=False); display(ablation_df)
seed_rows=[]
for s in SEEDS:
    z=D_hat.sample(frac=1,random_state=s); r=evaluate(f'DistilGPT2 seed {s}',pd.concat([D_major,z])[features],pd.concat([D_major,z]).Class); seed_rows.append({k:r[k] for k in ['F1','ROC-AUC','PR-AUC']})
seed_summary=pd.DataFrame(seed_rows).agg(['mean','std']).T.rename(columns={'mean':'Mean','std':'Std'}); display(seed_summary)
print('FINAL EXPERIMENT SUMMARY');
for _,r in result_df.iterrows(): print(f"{r.Method}: F1={r.F1:.4f}, ROC-AUC={r['ROC-AUC']:.4f}, PR-AUC={r['PR-AUC']:.4f}")
print('Synthetic Quality:',quality_metrics); print('Best F1:',result_df.loc[result_df.F1.idxmax(),'Method']); print('Best PR-AUC:',result_df.loc[result_df['PR-AUC'].idxmax(),'Method'])

## Research findings, limitations, and conclusion

The preceding cells generate findings from the actual metrics rather than hard-coded claims. Compare the proposed row with the baselines for F1, PR-AUC, ROC-AUC, recall, and precision; inspect `mean_ks`, approximate Close Probability/Coverage, DCR, PCA overlap, and the ablation table. The credit-card features are PCA-transformed numerical values, so interpolation is mathematically convenient but may not correspond to physically meaningful transactions. DistilGPT2 is a small autoregressive text model rather than the paper's exact model; parsing can reject malformed generations; debug mode uses fewer samples/epochs; and Close Probability/Coverage are explicitly approximations. The test set remains a final evaluation set only.

## How this implementation corresponds to the research paper

| Paper component | Notebook implementation | Exact/Approximate |
|---|---|---|
| Artificial minority imbalance | Stratified split, all training fraud as `D*_minor`, q=0.2 `D_minor` | Exact notation / dataset adaptation |
| `fix_y` | Class-first text and randomized feature order | Paper-inspired implementation |
| Interpolation | Uniform convex interpolation of numerical minority features | Approximation where formula/details are underspecified |
| LLM oversampling | DistilGPT2 causal LM fine-tuned on `D_minor + D_inter` | Approximate: model and practical settings differ |
| `condition_yx` | Class plus one observed minority feature/value prompt | Paper-inspired implementation |
| Close Probability/Coverage | Distance/radius definitions documented above | Approximate if paper's exact definition is unavailable |
| Downstream classifier | XGBoost, same untouched test set | Dataset/engineering adaptation |
| Reproducibility | fixed seeds, saved model, CSV/PNG outputs, debug switch | Exact operational requirement |

Run with `DEBUG_MODE=False` for the complete experiment and review the saved raw generations before interpreting quality metrics.